# Datathon 2026 Round 1 - Phase 4: MCQ
Notebook này chứa toàn bộ logic giải quyết 10 câu hỏi trắc nghiệm của Nhiệm vụ 1. Để đảm bảo tính nhất quán chéo môi trường, biến `SEED=42` được khai báo ở đầu file.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

base_path = '../dataset-datathon-2026-round-1/'

# Load data
orders = pd.read_csv(base_path + 'orders.csv', parse_dates=['order_date'])
products = pd.read_csv(base_path + 'products.csv')
returns = pd.read_csv(base_path + 'returns.csv')
web_traffic = pd.read_csv(base_path + 'web_traffic.csv')
order_items = pd.read_csv(base_path + 'order_items.csv', low_memory=False)
customers = pd.read_csv(base_path + 'customers.csv')
geography = pd.read_csv(base_path + 'geography.csv')
payments = pd.read_csv(base_path + 'payments.csv')

def format_vnd(value):
    """Format number to VND using comma as thousands separator (clear, no ambiguity)"""
    return f"{value:,.0f} VND"


Pandas version: 2.2.2
Numpy version: 1.26.4


## Q1. Trung vị số ngày giữa hai lần mua liên tiếp

In [2]:
multi_order_customers = orders['customer_id'].value_counts()
multi_order_customers = multi_order_customers[multi_order_customers > 1].index
multi_orders = orders[orders['customer_id'].isin(multi_order_customers)].sort_values(['customer_id', 'order_date'])
multi_orders['prev_order_date'] = multi_orders.groupby('customer_id')['order_date'].shift(1)
multi_orders['gap'] = (multi_orders['order_date'] - multi_orders['prev_order_date']).dt.days
q1_ans = multi_orders['gap'].median()
q1_choices = {30: 'A', 90: 'B', 180: 'C', 365: 'D'}
closest_q1 = min(q1_choices.keys(), key=lambda x: abs(x - q1_ans))
print(f"Q1: Median gap = {q1_ans} days -> CHỌN ĐÁP ÁN: {q1_choices[closest_q1]}")


Q1: Median gap = 144.0 days -> CHỌN ĐÁP ÁN: C


## Q2. Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất

In [3]:
products['margin'] = (products['price'] - products['cogs']) / products['price']
q2_ans = products.groupby('segment')['margin'].mean().idxmax()
q2_choices = {'Premium': 'A', 'Performance': 'B', 'Activewear': 'C', 'Standard': 'D'}
print(f"Q2: Highest margin segment = {q2_ans} -> CHỌN ĐÁP ÁN: {q2_choices.get(q2_ans, 'Không có trong đáp án')}")


Q2: Highest margin segment = Standard -> CHỌN ĐÁP ÁN: D


## Q3. Lý do trả hàng phổ biến nhất cho danh mục Streetwear

In [4]:
ret_prod = returns.merge(products, on='product_id')
streetwear_rets = ret_prod[ret_prod['category'] == 'Streetwear']
q3_ans = streetwear_rets['return_reason'].value_counts().idxmax()
q3_choices = {'defective': 'A', 'wrong_size': 'B', 'changed_mind': 'C', 'not_as_described': 'D'}
print(f"Q3: Most common return reason for Streetwear = {q3_ans} -> CHỌN ĐÁP ÁN: {q3_choices.get(q3_ans, 'Không có trong đáp án')}")


Q3: Most common return reason for Streetwear = wrong_size -> CHỌN ĐÁP ÁN: B


## Q4. Nguồn traffic có tỷ lệ thoát (bounce_rate) trung bình thấp nhất

In [5]:
q4_ans = web_traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin()
q4_choices = {'organic_search': 'A', 'paid_search': 'B', 'email_campaign': 'C', 'social_media': 'D'}
print(f"Q4: Lowest bounce rate traffic source = {q4_ans} -> CHỌN ĐÁP ÁN: {q4_choices.get(q4_ans, 'Không có trong đáp án')}")


Q4: Lowest bounce rate traffic source = email_campaign -> CHỌN ĐÁP ÁN: C


## Q5. Tỷ lệ phần trăm các dòng trong order_items có áp dụng khuyến mãi

In [6]:
promo_pct = (order_items['promo_id'].notnull().sum() / len(order_items)) * 100
q5_choices = {12: 'A', 25: 'B', 39: 'C', 54: 'D'}
closest_q5 = min(q5_choices.keys(), key=lambda x: abs(x - promo_pct))
print(f"Q5: Promo percentage = {promo_pct:.2f}% -> CHỌN ĐÁP ÁN: {q5_choices[closest_q5]}")


Q5: Promo percentage = 38.66% -> CHỌN ĐÁP ÁN: C


## Q6. Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất

In [7]:
valid_cust = customers[customers['age_group'].notnull()]
cust_orders = valid_cust.merge(orders, on='customer_id', how='left')
orders_per_group = cust_orders.groupby('age_group')['order_id'].nunique()
cust_per_group = valid_cust.groupby('age_group')['customer_id'].nunique()
q6_ans = (orders_per_group / cust_per_group).idxmax()
q6_choices = {'55+': 'A', '25-34': 'B', '35-44': 'C', '45-54': 'D'}
print(f"Q6: Age group with highest avg orders = {q6_ans} -> CHỌN ĐÁP ÁN: {q6_choices.get(q6_ans, 'Không có trong đáp án')}")


Q6: Age group with highest avg orders = 55+ -> CHỌN ĐÁP ÁN: A


## Q7. Vùng (region) tạo ra tổng doanh thu cao nhất

In [8]:
# Q7: GROSS revenue = quantity * unit_price (khớp với định nghĩa sales.csv['Revenue'])
oi_rev = order_items.assign(revenue=order_items['quantity'] * order_items['unit_price'])
oi_ord_geo = (
    oi_rev.merge(orders[['order_id', 'zip']], on='order_id')
          .merge(geography[['zip', 'region']], on='zip')
)
gross_revs = oi_ord_geo.groupby('region')['revenue'].sum().sort_values(ascending=False)

print("Q7 Gross Doanh thu các vùng (khớp định nghĩa sales.csv):")
for r, val in gross_revs.items():
    print(f"  - {r}: {format_vnd(val)}")

q7_choices = {'West': 'A', 'Central': 'B', 'East': 'C'}
if gross_revs.max() / gross_revs.min() < 1.05:
    print("Q7: Doanh thu xấp xỉ bằng nhau -> CHỌN ĐÁP ÁN: D")
else:
    q7_ans = gross_revs.idxmax()
    print(f"Q7: Region with highest revenue = {q7_ans} -> CHỌN ĐÁP ÁN: {q7_choices.get(q7_ans, 'Không có trong đáp án')}")


Q7 Gross Doanh thu các vùng (khớp định nghĩa sales.csv):
  - East: 7,637,532,676 VND
  - Central: 4,941,908,472 VND
  - West: 3,851,035,438 VND
Q7: Region with highest revenue = East -> CHỌN ĐÁP ÁN: C


## Q8. Phương thức thanh toán được sử dụng nhiều nhất trong các đơn huỷ

In [9]:
cancelled = orders[orders['order_status'] == 'cancelled']
q8_ans = cancelled['payment_method'].value_counts().idxmax()
q8_choices = {'credit_card': 'A', 'cod': 'B', 'paypal': 'C', 'bank_transfer': 'D'}
print(f"Q8: Most used payment method for cancelled = {q8_ans} -> CHỌN ĐÁP ÁN: {q8_choices.get(q8_ans, 'Không có trong đáp án')}")


Q8: Most used payment method for cancelled = credit_card -> CHỌN ĐÁP ÁN: A


## Q9. Kích thước (size) có tỷ lệ trả hàng cao nhất

In [10]:
# Q9: đề định nghĩa return rate = số bản ghi returns / số dòng order_items (tính theo rows)
item_prod = order_items.merge(products[['product_id', 'size']], on='product_id')
ret_prod2 = returns.merge(products[['product_id', 'size']], on='product_id')
size_items = item_prod['size'].value_counts()
size_returns = ret_prod2['size'].value_counts()
return_rates = (size_returns / size_items * 100).sort_values(ascending=False)
q9_ans = return_rates.idxmax()
q9_choices = {'S': 'A', 'M': 'B', 'L': 'C', 'XL': 'D'}
print(f"Q9: Return rate theo rows: {return_rates.round(3).to_dict()}")
print(f"Q9: Size with highest return rate = {q9_ans} ({return_rates[q9_ans]:.2f}%) -> CHỌN ĐÁP ÁN: {q9_choices.get(q9_ans, 'Không có trong đáp án')}")

# Cross-check bằng quantity/return_quantity để đề phòng đề diễn giải khác
size_units = item_prod.groupby('size')['quantity'].sum()
size_ret_units = ret_prod2.groupby('size')['return_quantity'].sum()
rate_by_units = (size_ret_units / size_units * 100).sort_values(ascending=False)
print(f"Q9 cross-check theo units: {rate_by_units.round(3).to_dict()} -> top = {rate_by_units.idxmax()}")


Q9: Return rate theo rows: {'S': 5.652, 'L': 5.625, 'M': 5.566, 'XL': 5.52}
Q9: Size with highest return rate = S (5.65%) -> CHỌN ĐÁP ÁN: A
Q9 cross-check theo units: {'S': 3.46, 'L': 3.428, 'M': 3.397, 'XL': 3.364} -> top = S


## Q10. Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất

In [11]:
# Q10: Installment plan với mean payment_value cao nhất (chỉ xét 1/3/6/12 theo đề)
payment_means = payments.groupby('installments')['payment_value'].mean()
q10_choices = {1: 'A', 3: 'B', 6: 'C', 12: 'D'}
payment_means_valid = payment_means.loc[[k for k in q10_choices if k in payment_means.index]]
print("Q10: Mean payment_value theo installments (chỉ 1/3/6/12):")
for k, v in payment_means_valid.sort_values(ascending=False).items():
    print(f"  - {k} kỳ: {format_vnd(v)}")
q10_ans = payment_means_valid.idxmax()
print(f"Q10: Installment plan with highest avg payment = {q10_ans} kỳ -> CHỌN ĐÁP ÁN: {q10_choices.get(q10_ans, 'Không có trong đáp án')}")


Q10: Mean payment_value theo installments (chỉ 1/3/6/12):
  - 6 kỳ: 24,447 VND
  - 3 kỳ: 24,400 VND
  - 12 kỳ: 24,246 VND
  - 1 kỳ: 24,113 VND
Q10: Installment plan with highest avg payment = 6 kỳ -> CHỌN ĐÁP ÁN: C


## Tổng hợp đáp án cuối cùng
Cell dưới in chuỗi 10 đáp án A/B/C/D để đội trưởng copy trực tiếp vào form nộp bài.

In [12]:
final_answers = {
    'Q1': q1_choices[closest_q1],
    'Q2': q2_choices.get(q2_ans, '?'),
    'Q3': q3_choices.get(q3_ans, '?'),
    'Q4': q4_choices.get(q4_ans, '?'),
    'Q5': q5_choices[closest_q5],
    'Q6': q6_choices.get(q6_ans, '?'),
    'Q7': q7_choices.get(q7_ans, '?'),
    'Q8': q8_choices.get(q8_ans, '?'),
    'Q9': q9_choices.get(q9_ans, '?'),
    'Q10': q10_choices.get(q10_ans, '?'),
}
print("=" * 50)
print("FINAL ANSWERS (copy vào form nộp bài):")
print("=" * 50)
for q, a in final_answers.items():
    print(f"  {q}: {a}")
print(f"\nChuỗi copy nhanh: {','.join(final_answers.values())}")


FINAL ANSWERS (copy vào form nộp bài):
  Q1: C
  Q2: D
  Q3: B
  Q4: C
  Q5: C
  Q6: A
  Q7: C
  Q8: A
  Q9: A
  Q10: C

Chuỗi copy nhanh: C,D,B,C,C,A,C,A,A,C
